In [2]:
import duckdb

In [4]:
con = duckdb.connect(database = 'dados_duckdb.db', read_only = False)

In [13]:
df = con.execute("""
SELECT *
FROM (
SELECT *,  ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
FROM bronze_produtos
WHERE data_ingestao >= '2026-09-30'
) WHERE row = 1""").fetchdf()
df.head(20)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-09-30 17:20:57.218016,1
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-09-30 17:20:57.218016,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-10-01 14:33:21.264564,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-10-01 14:33:21.264564,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-10-01 14:33:21.264564,1


In [16]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX': 'nm_produto', 'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor', 'LABST': 'vl_preco'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [18]:
df2 = df_final
df2 = df2.astype(
    {'id': 'int32', 
     'nm_produto': 'string',
     'id_categoria': 'string', 
     'id_fornecedor': 'int32', 
     'vl_preco': 'float32'
     }
    )
df2.dtypes
#df2.head(10)

id                 int32
nm_produto        string
id_categoria      string
id_fornecedor      int32
vl_preco         float32
dtype: object

In [19]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos(
    id BIGINT,
    nm_produto TEXT,
    id_categoria TEXT,
    id_fornecedor BIGINT,
    vl_preco FLOAT
)
""")

In [20]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [22]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [23]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)


,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [24]:
con.close()